In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import DBSCAN
import re
from sklearn.metrics import silhouette_score

In [2]:
data = pd.read_csv("/Users/nicolasbour/Desktop/Projets_IA/Projets_certif/bloc_03_Machine_Learning/03_the_north_face/Nico/sample_clean.csv")

In [3]:
data.head()

,id,description_texte,details,fabric,weight,pays
0,1,active classic boxers - s reason boxers cult f...,"silky capilene 1 fabric ultralight , breathabl...",3.7 - oz 100 % - recycled polyester gladiodor ...,99 g 3.5 oz,Mexico
1,2,active sport boxer briefs - skinning glory req...,"stretch mesh provides support , open - weave m...",4.6 - oz 93 % polyester 71 % recycled /7 % spa...,60 g 2.1 oz,Israel
2,3,active sport briefs - superbreathable - fly br...,"stretch mesh provides support , open - weave m...",4.6 - oz 93 % polyester 71 % recycled /7 % spa...,49 g 1.7 oz,Israel
3,4,"alpine guide pants - skin , climb ice , switch...",durable stretch - woven polyester 47 % recycle...,7.6 - oz 90 - denier 92 % polyester 47 % recyc...,597 g 20.7 oz,Vietnam
4,5,"alpine wind jkt - high ridges , steep ice alpi...","lightweight , breathable polyester ripstop fab...",shell 1.3 - oz 20 - denier 100 % polyester rip...,331 g 11.5 oz,China


In [7]:
data[["product", "description"]] = data["description_texte"].str.split(
    r"\s+-\s+", n=1, expand=True
).reset_index(drop=True)

data.drop(columns=["description_texte"], inplace=True)

In [8]:
data.head()

,id,details,fabric,weight,pays,product,description
0,1,"silky capilene 1 fabric ultralight , breathabl...",3.7 - oz 100 % - recycled polyester gladiodor ...,99 g 3.5 oz,Mexico,active classic boxers,"s reason boxers cult favorite - cool , especia..."
1,2,"stretch mesh provides support , open - weave m...",4.6 - oz 93 % polyester 71 % recycled /7 % spa...,60 g 2.1 oz,Israel,active sport boxer briefs,skinning glory requires movement boxers decidi...
2,3,"stretch mesh provides support , open - weave m...",4.6 - oz 93 % polyester 71 % recycled /7 % spa...,49 g 1.7 oz,Israel,active sport briefs,superbreathable - fly briefs minimalist s choi...
3,4,durable stretch - woven polyester 47 % recycle...,7.6 - oz 90 - denier 92 % polyester 47 % recyc...,597 g 20.7 oz,Vietnam,alpine guide pants,"skin , climb ice , switch rock , traverse knif..."
4,5,"lightweight , breathable polyester ripstop fab...",shell 1.3 - oz 20 - denier 100 % polyester rip...,331 g 11.5 oz,China,alpine wind jkt,"high ridges , steep ice alpine , jacket serves..."


In [9]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   id           500 non-null    int64 
 1   details      500 non-null    object
 2   fabric       500 non-null    object
 3   weight       492 non-null    object
 4   pays         492 non-null    object
 5   product      500 non-null    object
 6   description  500 non-null    object
dtypes: int64(1), object(6)
memory usage: 27.5+ KB


In [10]:
data.isnull().sum()

id             0
details        0
fabric         0
weight         8
pays           8
product        0
description    0
dtype: int64

In [11]:
data.nunique()

id             500
details        420
fabric         223
weight         243
pays            12
product        411
description    478
dtype: int64

In [ ]:
# # Lignes dont le label apparaît plusieurs fois
# doublons_label = data[data.duplicated(subset="label", keep=False)]
# display(doublons_label.sort_values("label"))

# # Lignes dont la description apparaît plusieurs fois
# doublons_description = data[data.duplicated(subset="description", keep=False)]
# display(doublons_description.sort_values("description"))

Les doublons montrent que certains produits ont le même label ou la même description. Cela ne semble pas indiquer une erreur mais plutôt que certains produits montrent des variations qui impliquent le même label ou la même description. Il n'y a aucun doublon exact, donc nous ne supprimerons pas de lignes.

In [13]:
def text_cleaning(text): 
    """Mettre en minuscules,
    Remplacer la ponctuation et les caractères spéciaux par des espaces, Supprimer les espaces en trop"""

    text = text.lower()
    text = re.sub(r"[\W_]+", " ", text)
    return text.strip()

data["pays"] = data["pays"].fillna("").apply(text_cleaning)

In [14]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_df=0.95,
    min_df=2
)

# Remplace les valeurs manquantes par une chaîne vide
matrice_tfidf = vectorizer.fit_transform(data["clean_text"].fillna(""))

print("Dimensions :", matrice_tfidf.shape)
print("Exemples de mots :", vectorizer.get_feature_names_out()[:20])

KeyError: 'clean_text'

In [ ]:
pd.DataFrame(
    matrice_tfidf[:5].toarray(),
    columns=vectorizer.get_feature_names_out()
)

In [ ]:
dbscan = DBSCAN(
    eps=0.5,
    min_samples=10,
    metric="cosine"
)

In [ ]:
data["cluster"] = dbscan.fit_predict(matrice_tfidf)

display(data["cluster"].value_counts().sort_index())

In [ ]:
display(data.sort_values("cluster"))

In [ ]:
print(f"Nombre de clusters : {len(set(data['cluster'])) - 1}")  # On soustrait 1 pour exclure le bruit (-1)

Le cluster -1 est le cluster considéré comme du bruit par l'algorithme Dbscan. Nous allons tenter de réduire le nombre de produits dans ce cluster en ajustant les paramètres de l'algorithme.

In [ ]:
parametres = {
    "eps": [0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    "min_samples": [2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 15]
}

resultats = []

for params in ParameterGrid(parametres):
    clusters = DBSCAN(**params, metric="cosine").fit_predict(matrice_tfidf)
    sans_bruit = clusters != -1
    try : 
        score = silhouette_score(
            matrice_tfidf[sans_bruit],
            clusters[sans_bruit],
            metric="cosine"
        )
    except ValueError:
        score = np.nan

    resultats.append({
        **params,
        "silhouette": score,
        "%_bruit": (1 - sans_bruit.mean()) * 100,
        "nb_clusters": len(set(clusters)) - (1 if -1 in clusters else 0)
    })

resultats = pd.DataFrame(resultats)
display(resultats.sort_values("nb_clusters", ascending=False))

In [ ]:
candidats = resultats[(resultats["nb_clusters"] <= 20) & (resultats["nb_clusters"] >= 10) & resultats["silhouette"].notna()]

In [ ]:
display(candidats)

In [ ]:
if candidats.empty:
    print("Aucun modèle valide avec au maximum 20 clusters.")
else:
    meilleur = candidats.sort_values("silhouette", ascending=False).iloc[0]
    display(meilleur)

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(resultats["%_bruit"], resultats["silhouette"])
plt.xlabel("Proportion de bruit")
plt.ylabel("Score de silhouette")
plt.show()

Ce modele montre un score de silhouette difficile à choisir puisque dans les clusters selctionnés, le score de silhouette est très faible. Nous allons travailler sur la qualité des données pour améliorer le score de silhouette.